In [30]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [31]:
directory = "E:/instagram_dataset_raws/Instagram influencer dataset/"

### Load influencers

In [32]:
import pandas as pd
import os

influencers = pd.read_csv(
    os.path.join(directory, "influencers.txt"),
    sep="\t",   # change to "," if needed
    header=None,
    names=["username", "category", "followers", "followees", "num_posts"]
)

influencers.head()


,username,category,followers,followees,num_posts
0,Username,Category,#Followers,#Followees,#Posts
1,==============================================...,NaN,NaN,NaN,NaN
2,makeupbynvs,beauty,1432,1089,363
3,jaquelinevandoski,beauty,137600,548,569
4,anisaartistry,beauty,64644,289,391


### Count how many influencers per category

In [33]:
influencers["category"].value_counts()


category
fashion        11911
other           5720
travel          4210
family          4070
food            3565
beauty          1542
interior        1195
fitness         1133
pet              587
Category           1
fasion             1
fashion 0.5        1
Name: count, dtype: int64

There are 33,935 influencers in 9 categories, and each influencer has 300 posts. Each post has metadata and image files. 
To reduce the data being used, I will do a random sample of 50 influencers from each category, and all 300 of their posts, resulting in 9x50x300=100k+ posts.
If this helps the performance, then I may increase the number of users used from the dataset.

OR, we do 300 influencers from each category, 50 of each of their posts. so 9*300*50 is 100k+

In [34]:
exclude_categories = [
    "Category",
    "fasion",
    "fashion 0.5"
]


In [35]:
N = 300

filtered_influencers = influencers[
    (~influencers["category"].isin(exclude_categories)) &
    (influencers["category"].notna())
].reset_index(drop=True)

sampled_influencers = (
    filtered_influencers
    .groupby("category", group_keys=False)
    .apply(lambda x: x.sample(
        n=min(len(x), N),  # avoid errors if category is small
        random_state=42
    ))
    .reset_index(drop=True)
)


C:\Users\sugar\AppData\Local\Temp\ipykernel_95136\769756936.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


In [36]:
sampled_influencers["category"].value_counts()


category
beauty      300
family      300
fashion     300
fitness     300
food        300
interior    300
other       300
pet         300
travel      300
Name: count, dtype: int64

### Build a set of valid usernames

In [37]:
valid_usernames = set(sampled_influencers["username"])


In [38]:
print(len(valid_usernames))

2700


### Build mapping from metadata JSON to image files, filtering for only selected influencers

In [39]:
mapping_file = os.path.join(directory, "JSON-Image_files_mapping.txt")

In [40]:
# Search for exact username and post id
target_username = "maddchadd"
target_post_id = "814722363889295440"

with open(mapping_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        if (
            line.startswith(target_username + "\t")
            and target_post_id in line
        ):
            print(f"Found at line {i}:")
            print(line)
            break
    else:
        print("No matching line found")

Found at line 5472020:
maddchadd	814722363889295440.info	['814722363889295440.jpg']



In [41]:
from collections import defaultdict
import ast

# json_to_images = defaultdict(list)
user_posts = defaultdict(list)

with open(mapping_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        parts = line.rstrip().split("\t")

        if len(parts) != 3:
            continue

        username, info_file, image_list_str = parts
        username = username.strip().lower()  # normalize
        if username not in valid_usernames:
            # print(f"excluded:{username}")
            continue

        # Safely parse "['img1.jpg', 'img2.jpg']"
        image_files = ast.literal_eval(image_list_str)

        # store raw post info temporarily
        user_posts[username].append((info_file, image_files))

        # # prepend username to filenames
        # full_info_file = f"{username}-{info_file}"
        # full_image_files = [f"{username}-{img}" for img in image_files]

        # json_to_images[full_info_file].extend(full_image_files)
        # json_to_images[full_info_file] = list(set(json_to_images[full_info_file]))


        if i % 100_000 == 0:
            print(f"Processed {i:,} lines")


Processed 900,000 lines
Processed 2,100,000 lines
Processed 3,100,000 lines
Processed 5,200,000 lines
Processed 5,300,000 lines
Processed 6,000,000 lines
Processed 8,200,000 lines
Processed 8,300,000 lines
Processed 8,400,000 lines
Processed 8,700,000 lines
Processed 9,000,000 lines


Randomly select 50 posts per user

In [42]:
import random

json_to_images = defaultdict(list)

for username, posts in user_posts.items():
    sampled_posts = random.sample(posts, min(50, len(posts)))

    for info_file, image_files in sampled_posts:
        full_info_file = f"{username}-{info_file}"
        full_image_files = [f"{username}-{img}" for img in image_files]

        json_to_images[full_info_file].extend(full_image_files)
        json_to_images[full_info_file] = list(set(json_to_images[full_info_file]))


### Verify mapping completion

In [43]:
print("Valid usernames:", len(valid_usernames))
print("Posts kept:", len(json_to_images))
print("Total images:", sum(len(v) for v in json_to_images.values()))


Valid usernames: 2700
Posts kept: 134875
Total images: 171415


In [44]:
len(json_to_images)

134875

In [45]:
next(iter(json_to_images.items()))

('0paline-2014775930511050102.info',
 ['0paline-2014775925838489241.jpg',
  '0paline-2014775925821784458.jpg',
  '0paline-2014775925838478446.jpg',
  '0paline-2014775925813521200.jpg',
  '0paline-2014775925846950483.jpg'])

Check images count

In [46]:
from collections import Counter

image_count_dist = Counter(len(v) for v in json_to_images.values())
image_count_dist


Counter({1: 120436,
         2: 6402,
         3: 3212,
         4: 1676,
         5: 997,
         6: 643,
         10: 532,
         7: 403,
         8: 322,
         9: 251,
         16: 1})

In [47]:
bad_post = [k for k, v in json_to_images.items() if len(v) > 10][0]
bad_post


'alligoldenphotography-1907860073340206212.info'

In [48]:
images = json_to_images[bad_post]

print("Number of images:", len(images))
for img in images:
    print(img)


Number of images: 16
alligoldenphotography-1907859653010782141.jpg
alligoldenphotography-1907858717915902378.jpg
alligoldenphotography-1907859662582177727.jpg
alligoldenphotography-1907859654092797370.jpg
alligoldenphotography-1907858710181681820.jpg
alligoldenphotography-1907858712782109408.jpg
alligoldenphotography-1907858710651276251.jpg
alligoldenphotography-1907859661391003119.jpg
alligoldenphotography-1907858714057241080.jpg
alligoldenphotography-1907859667766450920.jpg
alligoldenphotography-1907858713260092423.jpg
alligoldenphotography-1907859654839524992.jpg
alligoldenphotography-1907859670006171284.jpg
alligoldenphotography-1907859661852471508.jpg
alligoldenphotography-1907858718293272575.jpg
alligoldenphotography-1907858711389510592.jpg


### Whitelist of JSON filenames

In [49]:
json_files_to_load = set(json_to_images.keys())
len(json_files_to_load)

134875

In [50]:
next(iter(json_files_to_load))

'thelittlefarmhands-1982759599238074396.info'

### Functions to extract image paths and find files

In [51]:
def find_all_images(folder_path):
    return sorted([
        os.path.join(folder_path, f)
        for f in os.listdir(folder_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

In [52]:
def find_file_by_ext(folder_path, extensions):
    """
    Returns the first file in folder_path matching any extension in extensions
    """
    for f in os.listdir(folder_path):
        if f.lower().endswith(extensions):
            return os.path.join(folder_path, f)
    return None

### Quick view of metadata contents

In [53]:
import json, gzip, lzma

def load_metadata(path):
    if path.endswith(".gz"):
        with gzip.open(path, "rt", encoding="utf-8") as f:
            return json.load(f)
    elif path.endswith(".xz"):
        with lzma.open(path, "rt", encoding="utf-8") as f:
            return json.load(f)

# Test loading of metadata with 1 sample
json_dir = os.path.join(directory, "Post_metadata/posts_info/info")
# json_file = next(iter(json_files_to_load))
json_files_to_load_list = list(json_files_to_load)
json_file = json_files_to_load_list[100] # random entry

meta_path = os.path.join(json_dir, json_file)
# meta = load_metadata(meta_path)
with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
print(meta.keys())
# print(meta['node'].keys())
for k, v in meta.items(): #['node'].items():
    print("KEY:", k)
    print("VALUE:", v)
    print("-" * 40)

dict_keys(['gating_info', 'viewer_can_reshare', 'display_resources', 'viewer_in_photo_of_you', 'viewer_has_saved_to_collection', 'viewer_has_saved', 'owner', 'viewer_has_liked', 'id', 'edge_media_preview_like', 'edge_media_to_tagged_user', 'dimensions', 'fact_check_information', '__typename', 'location', 'shortcode', 'is_ad', 'caption_is_edited', 'edge_media_to_parent_comment', 'media_preview', 'taken_at_timestamp', 'edge_media_to_caption', 'tracking_token', 'has_ranked_comments', 'display_url', 'edge_web_media_to_related_media', 'edge_media_preview_comment', 'comments_disabled', 'edge_media_to_sponsor_user', 'accessibility_caption', 'is_video'])
KEY: gating_info
VALUE: None
----------------------------------------
KEY: viewer_can_reshare
VALUE: True
----------------------------------------
KEY: display_resources
VALUE: [{'src': 'https://scontent-lax3-1.cdninstagram.com/vp/9678824f22caf22e47c9e046ca690091/5E06792E/t51.2885-15/sh0.08/e35/s640x640/21227712_1888184171422856_58033039156508

### Extract post metadata from limited influencers

In [54]:
import json
import os
from pathlib import Path
from datetime import datetime, timezone
from tqdm import tqdm
from langdetect import detect, LangDetectException


records = []
bad_json_count = 0

json_dir = os.path.join(directory, "Post_metadata/posts_info/info")
image_dir = os.path.join(directory, "Post_images/posts_image/image")

# order and find total number
json_files_to_load = sorted(json_to_images.keys())
total_files = len(json_files_to_load)

# loop with progress 
for json_file in tqdm(json_files_to_load, total=total_files, desc="Processing posts"):
    json_path = os.path.join(json_dir, json_file)
    # json_path = str(Path(json_path).as_posix()) # normalise the path to /
    # if not os.path.exists(json_path):
    #     # print(json_path)
    #     continue

    try:
        with open(json_path, "r", encoding="utf-8") as f:
            metadata = json.load(f)
    except (json.JSONDecodeError, OSError, UnicodeDecodeError):
        bad_json_count += 1
        continue

    # ====================
    # JSON extraction
    # ====================

    # --- Extract metadata from nested JSON ---
    # node = metadata.get('node', {})

    # --- Caption ---
    edges = metadata.get("edge_media_to_caption", {}).get("edges", [])
    if not edges:
        continue

    caption = edges[0].get("node", {}).get("text", "").strip()
    if not caption:
        continue

    # --- Language filter ---
    try:
        if detect(caption) != "en": # keep only english captions
            continue
    except LangDetectException:
        continue

    # === Owner metrics ===
    # followers = metadata.get("owner", {}).get("edge_followed_by", {}).get("count", 0) #### DONT HAVE
    # following = metadata.get("owner", {}).get("edge_follow", {}).get("count", 0) #### DONT HAVE, in influencers.txt
    owner = metadata.get("owner", {})
    user_id = owner.get("id", None)#.get("count", 0)
    username = owner.get("username", None)

    # === Post metrics ===
    # Timestamp (convert from UNIX epoch to readable)
    ts = metadata.get('taken_at_timestamp', None)
    if ts:
        publish_timestamp = datetime.fromtimestamp(ts, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S')
    else:
        publish_timestamp = ''

    # Location
    has_location = 1 if metadata.get("location") else 0

    # # Flag carousel posts (> 1 image)
    # sidecar = metadata.get("edge_sidecar_to_children") #### DONT HAVE

    # is_carousel = (
    #     1 if sidecar and len(sidecar.get("edges", [])) > 0 else 0
    # )

    # # Get number of images in post
    # if sidecar and "edges" in sidecar:
    #     num_images = len(sidecar["edges"])
    # else:
    #     num_images = 1  # single-image post

    # Flag sponsored content
    sponsor_edge = metadata.get("edge_media_to_sponsor_user")

    is_sponsored = (
        1 if sponsor_edge and len(sponsor_edge.get("edges", [])) > 0 else 0
    )

    # === Engagement-related ===
    # likes = metadata.get("edge_media_preview_like", {}).get("count", 0)
    # comments = metadata.get("edge_media_preview_comment", {}).get("count", 0) ### NO 'TO_COMMENT', HAVE 'PREVIEW_COMMENT'
    preview_like = metadata.get("edge_media_preview_like", {})
    preview_comment = metadata.get("edge_media_preview_comment", {})

    likes = preview_like.get("count", 0)
    comments = preview_comment.get("count", 0)

    # Calculate engagement rate
    # engagement_rate = (likes + comments) / max(followers, 1) # avoid division by 0

    

    # --- Find Image / Video file ---
    # Filter out video posts for this project
    if metadata.get("is_video"):
        continue  # skip video posts

    # Find number of image files for post
    image_list = json_to_images.get(json_file, [])
    # Extract all images, save local relative path in list
    image_path = [
            os.path.join(image_dir, img)
            for img in image_list
        ]
    # images = find_all_images(image_dir)
    # images[0] if images else None

    
    # Flag carousel posts (>1 image)
    is_carousel = 1 if len(image_list) > 1 else 0
    # Count number of images in post
    num_images = len(image_list)

    # if not image_list:
    #     continue

    if image_path is None:
        continue  # skip posts without images

    # ===============================
    records.append({
        # Owner metrics
        "user_id": user_id,
        "username": username,
        # "followers": followers, # in influencers.txt
        # "following": following, # in influencers.txt
        # Post metrics
        "publish_timestamp": publish_timestamp,
        "has_location": has_location,
        "is_carousel": is_carousel,
        "num_images": num_images,
        "is_sponsored": is_sponsored,
        "image_path": image_path,
        "caption": caption,
        # Engagement-related
        "likes": likes,
        "comments": comments,
        # "engagement_rate": engagement_rate   # calculate later
    })

    # record = {
    #     "username": data.get("username"),
    #     "post_id": data.get("id"),
    #     "caption": data.get("caption"),
    #     "num_likes": data.get("likes"),
    #     "num_comments": data.get("comments"),
    #     "timestamp": data.get("timestamp"),
    #     "is_sponsored": data.get("sponsored", False),
    #     "image_paths": [
    #         os.path.join(image_dir, img)
    #         for img in json_to_images[json_file]
    #     ],
    #     "num_images": len(json_to_images[json_file])
    # }

    # records.append(record)
print("Bad JSON files skipped:", bad_json_count)

Processing posts: 100%|██████████| 134875/134875 [1:41:59<00:00, 22.04it/s] 

Bad JSON files skipped: 6239


### Create post-level dataframe

In [55]:
df_posts = pd.DataFrame(records)

df_posts.head()


,user_id,username,publish_timestamp,has_location,is_carousel,num_images,is_sponsored,image_path,caption,likes,comments
0,237173390,15minbeauty,2015-06-18 15:34:41,0,0,1,0,[E:/instagram_dataset_raws/Instagram influence...,#ad This summer I'm all about keeping my hair ...,39,4
1,237173390,15minbeauty,2015-06-22 12:37:40,0,0,1,0,[E:/instagram_dataset_raws/Instagram influence...,Trying out a ton of great new products right n...,98,0
2,237173390,15minbeauty,2015-07-06 15:33:50,0,0,1,0,[E:/instagram_dataset_raws/Instagram influence...,I spend a LOT of time and effort to keep my ha...,34,4
3,237173390,15minbeauty,2015-07-29 18:57:55,0,0,1,0,[E:/instagram_dataset_raws/Instagram influence...,"I'm not usually a shadow stick girl, they neve...",31,0
4,237173390,15minbeauty,2015-09-05 17:01:50,0,0,1,0,[E:/instagram_dataset_raws/Instagram influence...,#ad It's time to get ready for fall manicures ...,35,0


In [56]:
# Verify df_posts exists
df_posts.shape
# df_posts["num_images"].describe()
# df_posts["caption"].isna().mean()


(100432, 11)

### Merge influencer-level info (followers, followees) on username

In [57]:
df_posts_influencers = df_posts.merge(
    influencers[["username", "followers", "followees"]],
    on="username",
    how="left"
)

### Save dataframes as pickle

In [58]:
df_posts.to_pickle("df_posts_filtered.pkl")

In [59]:
df_posts_influencers.to_pickle("df_posts_influencers.pkl")